# Descriptive Statistics Tables

This notebook produces all descriptive statistics tables for the bachelor's thesis
on ABPM hemodynamic coupling.

In [ ]:
import sys
import os

# Ensure CWD is project root so Config relative paths resolve correctly
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

from abpm_hemodynamic_coupling.config import Config, Columns
from abpm_hemodynamic_coupling.data_processing import DataLoader
from analysis.thesis_stats import (
    format_median_iqr, format_mean_sd, derive_dipper_category,
    export_table, bootstrap_ci
)

config = Config()
THESIS_DIR = Path("results/thesis")
THESIS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300
sns.set_theme(style="whitegrid", font_scale=1.1)

In [ ]:
# Load all data sources
monitoring = pd.read_parquet(THESIS_DIR / "monitoring_labeled.parquet")
print(f"Monitoring: {monitoring.shape}")

ctx_medians = pd.read_csv(THESIS_DIR / "subject_context_medians.csv")
print(f"Subject-context medians: {ctx_medians.shape}")

agg = pd.read_csv(THESIS_DIR / "aggregated_clean.csv")
if "avg_reaction_time_battey_2" in agg.columns:
    agg = agg.rename(columns={"avg_reaction_time_battey_2": "avg_reaction_time_battery_2"})
print(f"Aggregated clean: {agg.shape}")

res = pd.read_csv(config.get_results_path(config.SUBJECT_METRICS_OUTPUT))
print(f"Pipeline results: {res.shape}")

## Table 1 --- Cohort Demographics

In [ ]:
# Variable definitions: (display_name, column, type)
# type: "continuous" -> auto-select mean+-SD or median(IQR) via Shapiro-Wilk
#       "categorical" -> n (%)

TABLE1_VARS = [
    ("Age, years", "age", "continuous"),
    ("Height, cm", "height_cm", "continuous"),
    ("Weight, kg", "weight_kilos", "continuous"),
    ("BMI, kg/m\u00b2", "bmi", "continuous"),
    ("Monitoring duration, min", "monitoring_duration_m", "continuous"),
    ("Sleep duration, h", "sleep_duration_h", "continuous"),
    ("BP load, %", "bp_load_%", "continuous"),
    ("SBP dip, %", "sbp_dip_%", "continuous"),
    ("DBP dip, %", "dbp_dip_%", "continuous"),
    ("Female sex", "is_female", "categorical"),
    ("Young (\u226430 y)", "is_young", "categorical"),
    ("Normal BMI", "bmi_is_normal", "categorical"),
    ("Healthy diagnosis", "monitoring_diagnosis_is_healthy", "categorical"),
    ("Chronic diseases", "has_chronic_diseases", "categorical"),
    ("Normal SBP dipper", "is_normal_dipper_by_sbp", "categorical"),
    ("Normal DBP dipper", "is_normal_dipper_by_dbp", "categorical"),
    ("Air alert during monitoring", "air_alert_during_monitoring", "categorical"),
    ("Sleep <7 h", "sleep_less_than_7h", "categorical"),
    ("Satisfied with sleep", "is_satisfied_with_sleep", "categorical"),
    ("Works under stress", "working_under_stress", "categorical"),
    ("Mental labor", "has_mental_labor", "categorical"),
    ("University degree", "has_university_degree", "categorical"),
    ("Registered partnership", "partnership_is_registered", "categorical"),
    ("Not smoking", "not_smoking", "categorical"),
    ("Drinks coffee", "drinks_coffee", "categorical"),
    ("Good food quality", "food_quality_is_good", "categorical"),
    ("Sleep before midnight", "sleep_before_midnight", "categorical"),
]


def describe_subgroup(df: pd.DataFrame, col: str, var_type: str) -> tuple[str, int]:
    """Return (formatted_string, n_missing) for a variable in a subgroup."""
    values = df[col].dropna()
    n_missing = int(df[col].isna().sum())
    n_valid = len(values)
    n_total = len(df)

    if var_type == "categorical":
        n_pos = int(values.sum())
        pct = n_pos / n_total * 100 if n_total > 0 else 0.0
        return f"{n_pos} ({pct:.1f}%)", n_missing

    # continuous: choose format based on Shapiro-Wilk
    if n_valid >= 3:
        _, p_val = stats.shapiro(values)
        if p_val > 0.05:
            return format_mean_sd(values), n_missing
    return format_median_iqr(values), n_missing


# Build table rows
male_mask = agg["is_female"] == 0
female_mask = agg["is_female"] == 1
n_male = int(male_mask.sum())
n_female = int(female_mask.sum())
n_total = len(agg)

rows = []
for display_name, col, var_type in TABLE1_VARS:
    overall_str, overall_miss = describe_subgroup(agg, col, var_type)
    male_str, _ = describe_subgroup(agg[male_mask], col, var_type)
    female_str, _ = describe_subgroup(agg[female_mask], col, var_type)
    rows.append({
        "Variable": display_name,
        f"Overall (N={n_total})": overall_str,
        f"Male (n={n_male})": male_str,
        f"Female (n={n_female})": female_str,
        "Missing": overall_miss,
    })

table1_df = pd.DataFrame(rows)
export_table(table1_df, THESIS_DIR / "table_01_cohort_demographics")
print("Exported table_01_cohort_demographics (.csv, .tex)")

In [ ]:
display(table1_df)

## Table 2 --- BP Summary by Monitoring Context

In [ ]:
CONTEXT_ORDER = [
    Columns.LABEL_BASELINE,
    Columns.LABEL_SLEEP,
    Columns.LABEL_BEFORE_TESTING,
    Columns.LABEL_COGNITIVE_PRE,
    Columns.LABEL_COGNITIVE_TASK,
    Columns.LABEL_PHYSICAL_PRE,
    Columns.LABEL_PHYSICAL_TASK,
    Columns.LABEL_BREAK,
    Columns.LABEL_AIR_ALERT,
]

t2_rows = []
for label in CONTEXT_ORDER:
    subset = ctx_medians[ctx_medians[Columns.LABEL] == label]
    n_subjects = subset["participant_id"].nunique()
    n_readings = int(subset["n"].sum())

    sbp_str = format_median_iqr(subset["SBP_med"]) if len(subset) > 0 else "---"
    dbp_str = format_median_iqr(subset["DBP_med"]) if len(subset) > 0 else "---"
    hr_str = format_median_iqr(subset["HR_med"]) if len(subset) > 0 else "---"

    t2_rows.append({
        "Context": label,
        "Subjects": n_subjects,
        "Readings": n_readings,
        "SBP, mmHg": sbp_str,
        "DBP, mmHg": dbp_str,
        "HR, bpm": hr_str,
    })

table2_df = pd.DataFrame(t2_rows)
export_table(table2_df, THESIS_DIR / "table_02_bp_by_context")
display(table2_df)
print("Exported table_02_bp_by_context (.csv, .tex)")

In [ ]:
# Context comparison box plots (subject-level medians)
plot_ctx = ctx_medians[ctx_medians[Columns.LABEL].isin(CONTEXT_ORDER)].copy()
plot_ctx[Columns.LABEL] = pd.Categorical(
    plot_ctx[Columns.LABEL], categories=CONTEXT_ORDER, ordered=True
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
measures = [("SBP_med", "SBP (mmHg)"), ("DBP_med", "DBP (mmHg)"), ("HR_med", "HR (bpm)")]

for ax, (col, ylabel) in zip(axes, measures):
    sns.boxplot(
        data=plot_ctx, x=Columns.LABEL, y=col, ax=ax,
        palette="Set2", fliersize=3,
    )
    sns.stripplot(
        data=plot_ctx, x=Columns.LABEL, y=col, ax=ax,
        color="black", size=3, alpha=0.4, jitter=0.2,
    )
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=8)

fig.suptitle("Subject-Level Median BP/HR by Monitoring Context", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_06_context_boxplots.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved fig_06_context_boxplots.png")

## Table 5 --- Dipping Status

In [ ]:
agg["sbp_dipper_cat"] = derive_dipper_category(agg["sbp_dip_%"])
agg["dbp_dipper_cat"] = derive_dipper_category(agg["dbp_dip_%"])

DIPPER_ORDER = ["Reverse dipper", "Non-dipper", "Normal dipper", "Extreme dipper"]
DIPPER_DEFS = {
    "Reverse dipper": "Dip < 0%",
    "Non-dipper": "0% \u2264 Dip < 10%",
    "Normal dipper": "10% \u2264 Dip < 20%",
    "Extreme dipper": "Dip \u2265 20%",
}

n_total = len(agg)
sbp_counts = agg["sbp_dipper_cat"].value_counts()
dbp_counts = agg["dbp_dipper_cat"].value_counts()

t5_rows = []
for cat in DIPPER_ORDER:
    sbp_n = int(sbp_counts.get(cat, 0))
    dbp_n = int(dbp_counts.get(cat, 0))
    t5_rows.append({
        "Category": cat,
        "Definition": DIPPER_DEFS[cat],
        "SBP n (%)": f"{sbp_n} ({sbp_n / n_total * 100:.1f}%)",
        "DBP n (%)": f"{dbp_n} ({dbp_n / n_total * 100:.1f}%)",
    })

table5_df = pd.DataFrame(t5_rows)
export_table(table5_df, THESIS_DIR / "table_05_dipping_status")
display(table5_df)
print("Exported table_05_dipping_status (.csv, .tex)")

## Table 6 --- Task Performance

In [ ]:
def _fmt_continuous(values: pd.Series) -> str:
    """Auto-select mean+-SD or median(IQR) based on Shapiro-Wilk."""
    clean = values.dropna()
    if len(clean) >= 3:
        _, p_val = stats.shapiro(clean)
        if p_val > 0.05:
            return format_mean_sd(clean)
    return format_median_iqr(clean)


t6_rows = []
for battery_num in [1, 2]:
    stroop_col = f"stroop_effect_battery_{battery_num}"
    neg_col = f"stroop_is_negative_battery_{battery_num}"
    rt_col = f"avg_reaction_time_battery_{battery_num}"

    stroop_str = _fmt_continuous(agg[stroop_col])

    neg_values = agg[neg_col].dropna()
    neg_n = int(neg_values.sum())
    neg_total = len(neg_values)
    neg_str = f"{neg_n} ({neg_n / neg_total * 100:.1f}%)" if neg_total > 0 else "---"

    rt_str = _fmt_continuous(agg[rt_col])

    # Count missing across all three columns
    n_missing = int(agg[[stroop_col, neg_col, rt_col]].isna().any(axis=1).sum())

    t6_rows.append({
        "Battery": f"Battery {battery_num}",
        "Stroop Effect": stroop_str,
        "Negative Stroop n (%)": neg_str,
        "Reaction Time": rt_str,
        "Missing": n_missing,
    })

table6_df = pd.DataFrame(t6_rows)
export_table(table6_df, THESIS_DIR / "table_06_task_performance")
display(table6_df)
print("Exported table_06_task_performance (.csv, .tex)")

## Summary

In [ ]:
artifacts = [
    "table_01_cohort_demographics.csv",
    "table_01_cohort_demographics.tex",
    "table_02_bp_by_context.csv",
    "table_02_bp_by_context.tex",
    "fig_06_context_boxplots.png",
    "table_05_dipping_status.csv",
    "table_05_dipping_status.tex",
    "table_06_task_performance.csv",
    "table_06_task_performance.tex",
]

print("Exported artifacts:")
for name in artifacts:
    path = THESIS_DIR / name
    status = "OK" if path.exists() else "MISSING"
    print(f"  [{status}] {path}")